In [ ]:
# -*- coding: utf-8 -*-
"""
generar_versiones.py - Genera las versiones v2 y v3 del dataset (simulan datos "actualizados").

Entrada : nodos.csv y aristas.csv (= version v1, los originales; no se modifican).
Salida  : nodos_v1/v2/v3.csv  y  aristas_v1/v2/v3.csv   + variacion_resumen.csv

Reglas (Opcion A):
  * nodos: v2 y v3 son copias EXACTAS de v1 (no tienen columnas numericas; asi node_id
    coincide siempre con las aristas).
  * aristas: v2 y v3 se generan a partir de v1 (NO una de otra), con semillas distintas, aplicando
    una variacion normal relativa:  valor_nuevo = valor_v1 x (1 + N(0, sigma)).
La semilla fija hace el resultado reproducible (mismos archivos cada vez que se ejecuta).

Uso (Colab o local, con nodos.csv y aristas.csv en la carpeta):
    python generar_versiones.py
"""
import os
import shutil

import numpy as np
import pandas as pd

NODOS, ARISTAS = "nodos.csv", "aristas.csv"
SEMILLAS = {2: 2, 3: 3}                 # una semilla por version

# Desviacion estandar RELATIVA (0.15 = 15 %). Documentar estos valores en el informe.
SIGMA = {
    "precio_moneda": 0.015,    # movimiento comun a todos los exchanges de una misma moneda
    "precio_fila": 0.003,      # diferencia propia de cada exchange (ruido pequeno)
    "withdrawal_fee": 0.15,    # comision de retiro (en la moneda)
    "taker_maker": 0.03,       # por exchange: Taker Fee y Maker Fee varian juntos
    "volumen": 0.25,           # Volumen 24h
}
COL_PRECIOS = ["Precio Last", "Precio Bid", "Precio Ask"]
COL_FEES = ["Taker Fee", "Maker Fee", "Withdrawal Fee", "Withdrawal Fee USD"]
COL_VARIADAS = COL_PRECIOS + COL_FEES + ["Volumen 24h"]


def _factor(rng, n, sigma):
    """Factor multiplicativo 1 + N(0, sigma), nunca menor a 0.05 (evita valores negativos)."""
    return np.clip(1.0 + rng.normal(0.0, sigma, n), 0.05, None)


def variar(df, semilla):
    rng = np.random.default_rng(semilla)
    out = df.copy()
    # Precios: un movimiento comun por moneda + un ruido pequeno por fila. El mismo factor se aplica
    # a Last, Bid y Ask, asi se conserva Bid <= Last <= Ask.
    monedas = out["Criptomoneda"].unique()
    f_moneda = dict(zip(monedas, _factor(rng, len(monedas), SIGMA["precio_moneda"])))
    f_precio = out["Criptomoneda"].map(f_moneda).to_numpy() * _factor(rng, len(out), SIGMA["precio_fila"])
    for c in COL_PRECIOS:
        out[c] = df[c] * f_precio
    # Comision de retiro: los ceros siguen en cero; el valor en USD se recalcula con el precio nuevo.
    out["Withdrawal Fee"] = df["Withdrawal Fee"] * _factor(rng, len(out), SIGMA["withdrawal_fee"])
    out["Withdrawal Fee USD"] = out["Withdrawal Fee"] * out["Precio Last"]
    # Taker y Maker: un factor por exchange.
    exchanges = out["Exchange"].unique()
    f_ex = out["Exchange"].map(dict(zip(exchanges, _factor(rng, len(exchanges), SIGMA["taker_maker"]))))
    out["Taker Fee"] = df["Taker Fee"] * f_ex
    out["Maker Fee"] = df["Maker Fee"] * f_ex
    out["Volumen 24h"] = df["Volumen 24h"] * _factor(rng, len(out), SIGMA["volumen"])
    return out


def verificar(v1, v, nombre):
    """Comprueba que la version sea coherente. Devuelve lista de problemas (vacia = todo bien)."""
    problemas = []
    if v.shape != v1.shape or list(v.columns) != list(v1.columns):
        problemas.append("la forma o las columnas cambiaron")
    for c in ["nodo_id", "Exchange", "Criptomoneda", "Red", "Retiro Activo", "Deposito Activo"]:
        if not v[c].equals(v1[c]):
            problemas.append(f"la columna {c} cambio")
    for c in COL_VARIADAS:
        if (v[c] < 0).any():
            problemas.append(f"hay valores negativos en {c}")
    for c in COL_FEES:
        if ((v1[c] == 0) & (v[c] != 0)).any():
            problemas.append(f"un cero de {c} dejo de ser cero")
    ok1 = ((v1["Precio Bid"] <= v1["Precio Last"]) & (v1["Precio Last"] <= v1["Precio Ask"])).sum()
    ok2 = ((v["Precio Bid"] <= v["Precio Last"]) & (v["Precio Last"] <= v["Precio Ask"])).sum()
    if ok2 < ok1:
        problemas.append("se rompio el orden Bid <= Last <= Ask")
    esperado = v["Withdrawal Fee"] * v["Precio Last"]
    if not np.allclose(v["Withdrawal Fee USD"].fillna(0), esperado.fillna(0), rtol=1e-9, atol=1e-12):
        problemas.append("Withdrawal Fee USD no es Withdrawal Fee x Precio Last")
    return problemas


def resumen(v1, v, version):
    filas = []
    for c in COL_VARIADAS:
        m = (v1[c] > 0) & v[c].notna()
        cambio = (v.loc[m, c] / v1.loc[m, c] - 1.0) * 100
        filas.append({"version": f"v{version}", "columna": c, "filas_variadas": int(m.sum()),
                      "cambio_medio_%": round(cambio.mean(), 3), "desviacion_estandar_%": round(cambio.std(), 3)})
    return filas


def main():
    if not (os.path.exists(NODOS) and os.path.exists(ARISTAS)):
        print(f"[!] Faltan {NODOS} y/o {ARISTAS} en esta carpeta.")
        return
    v1 = pd.read_csv(ARISTAS)
    faltan = [c for c in COL_VARIADAS + ["nodo_id", "Exchange", "Criptomoneda", "Red"] if c not in v1.columns]
    if faltan:
        print(f"[!] aristas.csv no tiene las columnas esperadas: {faltan}")
        return
    # v1 = originales (copia exacta con el nuevo nombre)
    shutil.copyfile(NODOS, "nodos_v1.csv")
    shutil.copyfile(ARISTAS, "aristas_v1.csv")
    # consistencia de v1 (el USD debe ser fee x precio, si no, el recalculo cambiaria la base)
    dif = (v1["Withdrawal Fee USD"].fillna(0) - (v1["Withdrawal Fee"] * v1["Precio Last"]).fillna(0)).abs()
    if (dif > 1e-9 + 1e-6 * v1["Withdrawal Fee USD"].abs().fillna(0)).any():
        print("[i] Aviso: en v1 algunas filas de Withdrawal Fee USD no son Withdrawal Fee x Precio Last; "
              "en v2 y v3 si lo seran.")
    todo = []
    for version, semilla in SEMILLAS.items():
        v = variar(v1, semilla)
        shutil.copyfile(NODOS, f"nodos_v{version}.csv")                   # Opcion A: nodos identicos
        v.to_csv(f"aristas_v{version}.csv", index=False)
        problemas = verificar(v1, v, f"v{version}")
        print(f"v{version}: aristas_v{version}.csv ({len(v):,} filas) | verificacion: "
              f"{'OK' if not problemas else 'PROBLEMAS -> ' + '; '.join(problemas)}")
        todo += resumen(v1, v, version)
    res = pd.DataFrame(todo)
    res.to_csv("variacion_resumen.csv", index=False)
    print("\nVariacion lograda (para el informe; la sigma objetivo esta en SIGMA al inicio del script):")
    print(res.to_string(index=False))


if __name__ == "__main__":
    main()

[!] Faltan nodos.csv y/o aristas.csv en esta carpeta.


In [ ]:
import pandas as pd

# Supongamos que este es tu DataFrame de aristas de la v2 y v3
for version in ['v2', 'v3']:
    df_aristas = pd.read_csv(f'aristas_{version}.csv')

    # Asegurar que las comisiones y precios no sean negativos ni absurdos
    cols_a_limpiar = ['Maker Fee', 'Taker Fee', 'Withdrawal Fee USD', 'Precio Last', 'Precio Bid', 'Precio Ask']
    for col in cols_a_limpiar:
        if col in df_aristas.columns:
            df_aristas[col] = df_aristas[col].clip(lower=0)

    # Guardar corregido en CSV y Parquet
    df_aristas.to_csv(f'aristas_{version}.csv', index=False)
    df_aristas.to_parquet(f'aristas_{version}.parquet', index=False)
    print(f"Aristas {version} corregidas y convertidas a Parquet sin valores negativos.")

Aristas v2 corregidas y convertidas a Parquet sin valores negativos.
Aristas v3 corregidas y convertidas a Parquet sin valores negativos.


In [ ]:
import os
import pandas as pd

# 1. Convertir todos los nodos (original, v1, v2, v3) a parquet
for nombre in ['nodos', 'nodos_v1', 'nodos_v2', 'nodos_v3']:
    if os.path.exists(f"{nombre}.csv"):
        df = pd.read_csv(f"{nombre}.csv")
        df.to_parquet(f"{nombre}.parquet", index=False)
        print(f"Convertido exitosamente: {nombre}.parquet")

# 2. Convertir la arista v1 (y la arista original si la usas como fallback) a parquet
for nombre in ['aristas', 'aristas_v1']:
    if os.path.exists(f"{nombre}.csv"):
        df = pd.read_csv(f"{nombre}.csv")
        df.to_parquet(f"{nombre}.parquet", index=False)
        print(f"Convertido exitosamente: {nombre}.parquet")

Convertido exitosamente: nodos.parquet
Convertido exitosamente: nodos_v1.parquet
Convertido exitosamente: nodos_v2.parquet
Convertido exitosamente: nodos_v3.parquet
Convertido exitosamente: aristas.parquet
Convertido exitosamente: aristas_v1.parquet
